<a href="https://colab.research.google.com/github/shihas-36/Multimodal-Depression-Detection/blob/main/ORG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **MODULE 1**

In [ ]:
import pandas as pd

In [ ]:

from sklearn.model_selection import train_test_split
!pip install pandas scikit-learn transformers torch
!pip install openpyxl

In [ ]:
# --- 1. Load the Data ---
try:
    df = pd.read_excel('/content/drive/MyDrive/depjct/Sentiment Analysis Dataset 2..xlsm')
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: 'dataset.xlsx' not found. Please upload the file to your Colab environment or Google Drive.")
except Exception as e:
    print(f"An error occurred: {e}")
    df = None

In [ ]:
if df is not None:
    # Displaying the first few rows and column names to confirm successful loading
    print("\nInitial Data:")
    print(df.head())
    print("\nDataset shape:", df.shape)

    # --- 2. Identify Inputs and Labels ---
    # The dataset has 'text' and 'label' columns.
    # 'text' column contains the social media posts.
    # 'label' column contains the depression score (our target variable).
    X = df['text']
    y = df['label']
    print("\nSuccessfully identified input features (X) and labels (y).")


In [ ]:
# --- Data Cleaning: Drop Unnamed Columns ---
# Based on the initial data inspection, there are several unnamed columns that seem to be empty.
# We will drop these columns as they are not relevant for the sentiment analysis task.
unnamed_cols = [col for col in df.columns if 'Unnamed:' in col]
if unnamed_cols:
    df = df.drop(columns=unnamed_cols)
    print(f"Dropped columns: {unnamed_cols}")
    print("\nDataset shape after dropping unnamed columns:", df.shape)
    display(df.head())
else:
    print("No 'Unnamed:' columns found to drop.")

In [ ]:
    # --- 2.5. Data Cleaning (New Step to fix the error) ---
    # The ValueError indicates that the label column 'y' contains NaN values.
    # We must remove these rows before splitting the data.
    original_size = len(df)
    df.dropna(subset=['label'], inplace=True)
    cleaned_size = len(df)
    print(f"\nRemoved {original_size - cleaned_size} rows with missing labels.")
    print("\nDataset shape after cleaning:", df.shape)


In [ ]:
# --- Create a Balanced and Trimmed Dataset ---
# Creat a new dataframe 'df_trim' with an equal number of samples for each label (0 and 1).
# The target total size for the trimmed dataset is 50,000 samples (25,000 for each label).

if df is not None and 'label' in df.columns:
    # Separate samples by label
    df_label_0 = df[df['label'] == 0]
    df_label_1 = df[df['label'] == 1]

    # Determine the number of samples to take from each label
    # We want 25,000 for each, or fewer if there aren't enough samples in the original data
    samples_per_label = min(25000, len(df_label_0), len(df_label_1))

    if samples_per_label < 25000:
        print(f"Warning: Not enough samples to create a dataset of 50,000 with 25,000 per label.")
        print(f"Creating a balanced dataset with {samples_per_label} samples per label, total size {samples_per_label * 2}.")


    # Randomly sample from each label
    df_trim_0 = df_label_0.sample(n=samples_per_label, random_state=42)
    df_trim_1 = df_label_1.sample(n=samples_per_label, random_state=42)

    # Concatenate the sampled dataframes to create the trimmed dataset
    df_trim = pd.concat([df_trim_0, df_trim_1])

    # Shuffle the trimmed dataset to mix the labels
    df_trim = df_trim.sample(frac=1, random_state=42).reset_index(drop=True)

    print("\nTrimmed and balanced dataset created (df_trim).")
    print("Dataset shape after trimming and balancing:", df_trim.shape)
    print("\nLabel Distribution in df_trim:")
    print(df_trim['label'].value_counts())
    display(df_trim.head())

else:
    print("\nCannot create trimmed dataset. DataFrame is not loaded or 'label' column is missing.")

In [ ]:
df_og=df
df=df_trim

# Invert the labels: 0 becomes 1, and 1 becomes 0
df['label'] = 1 - df['label']
print("Labels in df_trim inverted (0 becomes 1, 1 becomes 0).")

In [ ]:
df_trim

In [ ]:


    # Re-assign X and y with the cleaned data
    X = df['text']
    y = df['label']
    print("\nDataset shape after cleaning:", df.shape)


In [ ]:
# --- 3. Data Splitting ---
# Split the data into a training set, a validation set, and a test set.
# We use a 3-way split to ensure robust evaluation.
# The 'stratify' parameter ensures the proportion of labels is the same in all splits,
# which is important for balanced classification.
# First, split the data into a training+validation set and a test set (e.g., 80% train+val, 20% test)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Next, split the training+validation set into a final training set and a validation set
# This creates a 60/20/20 split for train/validation/test from the original dataset
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25,  # 0.25 of 0.8 is 0.2
    random_state=42,
    stratify=y_train_val
)

print("\nData splitting complete.")
print(f"Training set size: {len(X_train)} samples")
print(f"Validation set size: {len(X_val)} samples")
print(f"Test set size: {len(X_test)} samples")

In [ ]:
import pandas as pd
import re
from transformers import BertTokenizer

In [ ]:
# --- 1. Define a placeholder for the data from Step 1 ---
# In your actual Colab notebook, you would use the X_train, X_val, X_test
# --- 2. Text Cleaning Function ---
# This function performs standard cleaning operations to prepare the text.
def clean_text(text):
    """
    Cleans the input text by removing special characters, extra spaces, and converting to lowercase.
    """
    # Convert text to string type to avoid errors with NaN values
    if not isinstance(text, str):
        return ""
    # Convert to lowercase
    text = text.lower()
    # Remove special characters, URLs, and mentions (adjusting based on the dataset)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Apply the cleaning function to the training data


In [ ]:

# Apply the cleaning function to the training data
X_train_cleaned = X_train.apply(clean_text)

print("Original Text:")
print(X_train.iloc)
print("\nCleaned Text:")
print(X_train_cleaned.iloc)


In [ ]:
# --- 3. Tokenization using a Pre-trained Transformer ---
# This is a crucial step for preparing data for a Transformer model.
# We will use a tokenizer that aligns with a pre-trained model like BERT.
# First, you would need to install the Hugging Face Transformers library in your Colab notebook:
#!pip install transformers
# Original: tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# Original: model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)
# Change to a model like RoBERTa:
# from transformers import RobertaForSequenceClassification, RobertaTokenizer

# Change to TwHIN-BERT
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Load the tokenizer for the pre-trained TwHIN-BERT model.
# The tokenizer uses a subword method like WordPiece or BPE, which are effective
# for handling out-of-vocabulary words and are fundamental to Transformer models. [1]
tokenizer = AutoTokenizer.from_pretrained('Twitter/twhin-bert-base')

# Tokenize the cleaned training data.
# We set max_length to a value that fits the model's requirements (e.g., 128 or 256).
# 'padding=True' ensures all sequences have the same length by adding special tokens.
# 'truncation=True' cuts off sentences longer than max_length.
tokenized_inputs = tokenizer(
    X_train_cleaned.tolist(),
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'  # Returns PyTorch tensors for model input
)

print("\n--- Tokenization Complete ---")
print("Example of a tokenized input (input_ids):")
print(tokenized_inputs['input_ids'])
print("\nExample of an attention mask (distinguishes real tokens from padding):")
print(tokenized_inputs['attention_mask'])


In [ ]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader, RandomSampler
from transformers import BertForSequenceClassification, BertTokenizer
from torch.optim import AdamW

In [ ]:
# In your actual Colab notebook, you would use the real tokenized_inputs and y_train
# you created. This is a placeholder to make the code runnable for demonstration.
# The 'Students anxiety and depression dataset' has two labels, 0 or 1.[1]
# The number of labels should match your dataset.
# The input_ids are from the tokenization step.
input_ids = tokenized_inputs['input_ids']
attention_mask = tokenized_inputs['attention_mask']
# We will assume a small number of samples for demonstration purposes.
y_train

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:

model = AutoModelForSequenceClassification.from_pretrained('Twitter/twhin-bert-base', num_labels=2)
model.to(device)

In [ ]:

# Define the optimizer and training parameters.
# AdamW is a highly effective optimizer, a variant of ADAM, which is a good choice for deep learning.
optimizer = AdamW(model.parameters(), lr=2e-5)
epochs = 3
batch_size = 32



In [ ]:
# The y_train variable is a pandas.Series and needs to be converted to a PyTorch tensor.
# --- THE FIX for the AssertionError ---
# Convert the pandas Series of labels to a PyTorch tensor.
y_train_tensor = torch.tensor(y_train.tolist(), dtype=torch.long)
y_val_tensor = torch.tensor(y_val.tolist(), dtype=torch.long)


# Create PyTorch datasets and dataloaders for efficient training.
train_dataset = TensorDataset(tokenized_inputs['input_ids'], tokenized_inputs['attention_mask'], y_train_tensor)
train_dataloader = DataLoader(train_dataset, sampler=RandomSampler(train_dataset), batch_size=batch_size)

In [ ]:
# @title
from tqdm.auto import tqdm # <-- ADD THIS IMPORT

In [ ]:
# --- 3. Model Fine-Tuning (The Training Loop) ---
# This is the core of the model trainer component.
model.train()  # Set the model to training mode
for epoch in range(epochs):
    print(f"\n===== Fine-tuning Epoch {epoch + 1}/{epochs} =====")
    total_loss = 0

    # Wrap the inner loop with tqdm to get a progress bar
    for batch in tqdm(train_dataloader, desc=f"Training Epoch {epoch + 1}"):
        # Move the batch tensors to the GPU
        b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

        # Clear any previously calculated gradients
        model.zero_grad()

        # Forward pass: Feed the inputs to the model.
        outputs = model(
            b_input_ids,
            attention_mask=b_attention_mask,
            labels=b_labels
        )

        # Get the loss from the model's output
        loss = outputs.loss
        total_loss += loss.item()

        # Backward pass: Calculate gradients for backpropagation
        loss.backward()

        # Update model parameters using the optimizer
        optimizer.step()

    avg_train_loss = total_loss / len(train_dataloader)
    print(f"  Average training loss: {avg_train_loss:.4f}")

print("\n--- Fine-tuning complete. ---")



In [ ]:

# --- CLEAR THE GPU CACHE ---
# This is a crucial step to free up reserved but unused memory after the training process.
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("CUDA cache cleared to free up VRAM.")


In [ ]:
# --- 1. Define a placeholder for the data from Step 1 ---
# In your actual Colab notebook, you would use the X_train, X_val, X_test
# --- 2. Text Cleaning Function ---
# This function performs standard cleaning operations to prepare the text.
# Apply the cleaning function to the training data
X_val_cleaned = X_val.apply(clean_text)

print("Original Text:")
print(X_val.iloc)
print("\nCleaned Text:")
print(X_val_cleaned.iloc)



In [ ]:
from torch.utils.data import DataLoader, TensorDataset

# Tokenize in smaller chunks
val_encodings = tokenizer(
    X_val_cleaned.tolist(),
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'
)

dataset = TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'])
dataloader = DataLoader(dataset, batch_size=16)  # reduce batch_size if still OOM

model.eval()
feature_vectors = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask = [b.to(device) for b in batch]
        outputs = model(input_ids, attention_mask=attention_mask, output_hidden_states=True)
        batch_features = outputs.hidden_states[-1][:, 0, :].cpu()
        feature_vectors.append(batch_features)

# Concatenate all features
import torch
feature_vectors = torch.cat(feature_vectors, dim=0)
print(feature_vectors.shape)


In [ ]:
feature_vectors.shape
print("\n--- Feature Extraction Complete ---")
print("Example of a final feature vector (embedding) ready for Module 3:")
print(feature_vectors.shape) # Should be  for bert-base
print(feature_vectors)

# This feature vector, along with the corresponding label, is the final output of Module 1.
# It is now ready to be fused with the wearable data features in Module 3.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from torch.utils.data import SequentialSampler

# --- 3.3 Model Evaluation on the Validation Set ---

print("\n\n===== Model Evaluation =====")

# Create a DataLoader for the validation set.
val_dataset = TensorDataset(val_encodings['input_ids'], val_encodings['attention_mask'], y_val_tensor)
val_dataloader = DataLoader(val_dataset, sampler=SequentialSampler(val_dataset), batch_size=16)

# Set the model to evaluation mode.
model.eval()

predictions = []
true_labels = []

# Wrap the loop with tqdm to monitor progress.
for batch in tqdm(val_dataloader, desc="Evaluating on Validation Set"):
    b_input_ids, b_attention_mask, b_labels = [t.to(device) for t in batch]

    with torch.no_grad():
        outputs = model(b_input_ids, attention_mask=b_attention_mask)

    # Get the predicted class (the one with the highest logit).
    logits = outputs.logits.detach().cpu().numpy()
    label_ids = b_labels.to('cpu').numpy()

    predictions.append(logits.argmax(axis=1))
    true_labels.append(label_ids)

# Concatenate the results from all batches.
flat_predictions = [item for sublist in predictions for item in sublist]
flat_true_labels = [item for sublist in true_labels for item in sublist]

# Calculate the metrics.
accuracy = accuracy_score(flat_true_labels, flat_predictions)
precision = precision_score(flat_true_labels, flat_predictions, average='weighted', zero_division=0)
recall = recall_score(flat_true_labels, flat_predictions, average='weighted', zero_division=0)
f1 = f1_score(flat_true_labels, flat_predictions, average='weighted', zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

# --- 4. Verification of the Final Output (Feature Vector) ---
# The code for feature extraction will output a tensor.
# To check if it's correct, verify its shape and data type.
# The `torch.Size()` output you received is exactly what's expected.
# You should also ensure the output is a PyTorch tensor on the CPU (`.cpu()`).

In [ ]:
# --- 1. Define a placeholder for the data from Step 1 ---
# In your actual Colab notebook, you would use the X_train, X_val, X_test
# --- 2. Text Cleaning Function ---
# This function performs standard cleaning operations to prepare the text.
# Apply the cleaning function to the training data
X_val_cleaned = X_val.apply(clean_text)

print("Original Text:")
print(X_val.iloc)
print("\nCleaned Text:")
print(X_val_cleaned.iloc)



In [ ]:
# --- Manual Input Check ---
# This section allows you to input text manually and see the model's prediction.

# Get input from the user
manual_input_text = input("Enter text to check for depression risk: ")

# Clean the input text using the same cleaning function
cleaned_input_text = clean_text(manual_input_text)

# Tokenize the cleaned input text
input_encodings = tokenizer(
    cleaned_input_text,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'  # Returns PyTorch tensors
)

# Move tensors to the appropriate device (GPU if available)
input_ids = input_encodings['input_ids'].to(device)
attention_mask = input_encodings['attention_mask'].to(device)

# Set the model to evaluation mode
model.eval()

# Make a prediction
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)

# Get the predicted label (0 or 1)
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

# Interpret the prediction (original 1 is now 0, original 0 is now 1)
if predicted_label == 1:
    print("\nBased on the input text, the model predicts a higher risk of depression.")
else:
    print("\nBased on the input text, the model predicts a lower risk of depression.")

In [ ]:
import numpy as np
import pandas as pd
import torch
import re
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from torch.utils.data import TensorDataset, DataLoader, SequentialSampler
from tqdm.auto import tqdm # Used for the progress bar
from transformers import AutoTokenizer, AutoModelForSequenceClassification # Import necessary classes

# Assuming 'tokenizer' and 'model' are already defined and loaded from previous cells.
# Assuming 'X_val_cleaned' and 'y_val_numpy' are already defined from previous cells.
# Assuming 'device' is already defined from previous cells.


# --- 2. Data Preparation for Feature Extraction ---
# Ensure X_val_cleaned and y_val_numpy are prepared before this cell runs.
X_val_cleaned = X_val.apply(clean_text) # Assuming clean_text is defined
y_val_numpy = y_val.values # Assuming y_val is defined


# Tokenize the cleaned validation data
val_encodings = tokenizer(
    X_val_cleaned.tolist(),
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'
)


# Concatenate all extracted features
# The variable 'feature_vectors' is already a concatenated tensor from the previous cell (anbo6lp9BVwO).
# Trying to evaluate 'if feature_vectors:' on a multi-element tensor causes a RuntimeError.
# We will explicitly check if the tensor has any elements.
if feature_vectors is not None and feature_vectors.numel() > 0:
    # No need to re-concatenate feature_vectors_tensor = torch.cat(feature_vectors, dim=0)
    # as feature_vectors is already the concatenated tensor.
    X_tsne = feature_vectors.numpy()
    print(f"Feature Extraction Complete. Extracted feature shape: {X_tsne.shape}")
else:
    raise ValueError("No features were extracted. Check your DataLoader and model.")

# --- 4. Apply t-SNE Transformation ---
print("Applying t-SNE to reduce to 2D...")
tsne = TSNE(n_components=2, random_state=42, verbose=1, n_jobs=-1, perplexity=30)
X_tsne_2d = tsne.fit_transform(X_tsne)

# --- 5. Plotting the t-SNE Results ---
tsne_df = pd.DataFrame(data=X_tsne_2d, columns=['tsne_1', 'tsne_2'])
tsne_df['label'] = y_val_numpy

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    tsne_df['tsne_1'],
    tsne_df['tsne_2'],
    c=tsne_df['label'],
    cmap='RdYlBu',
    alpha=0.6,
    s=10
)

# Add Legend and Labels
# Labels are now inverted: original 0 (Low Risk) is now 1, original 1 (High Risk) is now 0
legend_labels = {0: 'High Risk (0)', 1: 'Low Risk (1)'}
legend_handles = scatter.legend_elements()[0]

# --- FIX for the KeyError ---
# Extract the integer labels from the LaTeX formatted strings for legend lookup
legend_keys = [int(re.search(r'\d+', key).group()) for key in scatter.legend_elements()[1]]

plt.legend(legend_handles, [legend_labels[key] for key in legend_keys],
           loc="lower left", title="Depression Risk")

plt.title('t-SNE Visualization of TwHIN-BERT Embeddings (Module 1 - Inverted Labels)')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.savefig('tsne_visualization_module1_inverted.png')
plt.show()
print("t-SNE visualization complete and saved as 'tsne_visualization_module1_inverted.png'.")

In [ ]:
# --- Manual Input Check ---
# This section allows you to input text manually and see the model's prediction.

# Get input from the user
manual_input_text = input("Enter text to check for depression risk: ")

# Clean the input text using the same cleaning function
cleaned_input_text = clean_text(manual_input_text)

# Tokenize the cleaned input text
input_encodings = tokenizer(
    cleaned_input_text,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'  # Returns PyTorch tensors
)

# Move tensors to the appropriate device (GPU if available)
input_ids = input_encodings['input_ids'].to(device)
attention_mask = input_encodings['attention_mask'].to(device)

# Set the model to evaluation mode
model.eval()

# Make a prediction
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)

# Get the predicted label (0 or 1)
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

# Interpret the prediction
if predicted_label == 0:
    print("\nBased on the input text, the model predicts a higher risk of depression.")
else:
    print("\nBased on the input text, the model predicts a lower risk of depression.")

In [ ]:
# --- Manual Input Check ---
# This section allows you to input text manually and see the model's prediction.

# Get input from the user
manual_input_text = input("Enter text to check for depression risk: ")

# Clean the input text using the same cleaning function
cleaned_input_text = clean_text(manual_input_text)

# Tokenize the cleaned input text
input_encodings = tokenizer(
    cleaned_input_text,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'  # Returns PyTorch tensors
)

# Move tensors to the appropriate device (GPU if available)
input_ids = input_encodings['input_ids'].to(device)
attention_mask = input_encodings['attention_mask'].to(device)

# Set the model to evaluation mode
model.eval()

# Make a prediction
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)

# Get the predicted label (0 or 1)
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

# Interpret the prediction
if predicted_label == 0:
    print("\nBased on the input text, the model predicts a higher risk of depression.")
else:
    print("\nBased on the input text, the model predicts a lower risk of depression.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd
import torch

# Assuming 'feature_vectors' is the tensor of extracted features from Module 1
# If feature_vectors is a list of tensors, concatenate them first
if isinstance(feature_vectors, list):
    feature_vectors_tensor = torch.cat(feature_vectors, dim=0)
else:
    feature_vectors_tensor = feature_vectors

feature_vectors_numpy = feature_vectors_tensor.numpy()

# Assuming 'y_val' contains the corresponding labels for the validation set
# Convert y_val (pandas Series) to numpy array if it's not already
y_val_numpy = y_val.values

print("--- Applying PCA for Visualization ---")

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2, random_state=42)
X_pca_2d = pca.fit_transform(feature_vectors_numpy)

# Create a DataFrame for plotting
pca_df = pd.DataFrame(data=X_pca_2d, columns=['pca_1', 'pca_2'])
pca_df['label'] = y_val_numpy

# Plotting the PCA results
plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    pca_df['pca_1'],
    pca_df['pca_2'],
    c=pca_df['label'],
    cmap='RdYlBu', # Using a divergent color map
    alpha=0.6,
    s=10
)

# Add Legend and Labels
# Labels are now inverted: original 0 (Low Risk) is now 1, original 1 (High Risk) is now 0
legend_labels = {0: 'High Risk (0)', 1: 'Low Risk (1)'}
legend_handles, _ = scatter.legend_elements() # Get legend handles and labels

# Ensure legend labels match the data
# If scatter.legend_elements()[1] is not directly usable as keys,
# you might need to map them to the actual numerical labels (0 and 1)
# based on the scatter plot's internal mapping or the order of appearance.
# For simplicity, assuming the order corresponds to sorted unique labels:
unique_labels_in_plot = sorted(pca_df['label'].unique())
mapped_legend_labels = [legend_labels[label] for label in unique_labels_in_plot]


plt.legend(legend_handles, mapped_legend_labels,
           loc="lower left", title="Depression Risk")


plt.title('PCA Visualization of TwHIN-BERT Embeddings (Module 1 - Inverted Labels)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.savefig('PCA_visualization_module1_inverted.png')

plt.show()

print("\nPCA visualization complete.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import pandas as pd
import torch

# Assuming 'X_wearable_scaled' is the numpy array of scaled wearable features from Module 2
# Assuming 'y_wearable' contains the corresponding labels for the wearable data

# 1. Apply PCA Transformation
print("--- Applying PCA for Visualization (Module 2) ---")

# Apply PCA to reduce to 2 dimensions
pca = PCA(n_components=2, random_state=42)
X_pca_2d_wearable = pca.fit_transform(X_wearable_scaled)

# 2. Plotting the PCA Results
pca_df_wearable = pd.DataFrame(data=X_pca_2d_wearable, columns=['pca_1', 'pca_2'])
pca_df_wearable['label'] = y_wearable.values # Use y_wearable for labels

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    pca_df_wearable['pca_1'],
    pca_df_wearable['pca_2'],
    c=pca_df_wearable['label'],
    cmap='RdYlBu', # Using a divergent color map
    alpha=0.6,
    s=10
)

# Add Legend and Labels
legend_labels = {0: 'Low Risk (0)', 1: 'High Risk (1)'}
legend_handles, _ = scatter.legend_elements() # Get legend handles and labels

# Ensure legend labels match the data
unique_labels_in_plot = sorted(pca_df_wearable['label'].unique())
mapped_legend_labels = [legend_labels[label] for label in unique_labels_in_plot]


plt.legend(legend_handles, mapped_legend_labels,
           loc="lower left", title="Depression Risk")

plt.title('PCA Visualization of Aggregated Wearable Features (Module 2)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.grid(True)
plt.savefig('PCA_visualization_module2.png')
plt.show()

print("\nPCA visualization complete for Module 2.")

# **MODULE 2**

In [ ]:
# Cell 1: Setup, Ingestion, and Robust Imputation

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import warnings
import re
from torch.utils.data import TensorDataset, DataLoader # Needed for LSTM batching

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore', category=UserWarning)

# --- Define File Path ---
file_path = '/content/drive/MyDrive/depjct/sensor_hrv_filtered.csv'


try:
    df = pd.read_csv(file_path)
    print("--- Part 1: Data Ingestion & Initial Setup ---")

    # 1. Clean Feature Names
    def clean_col_name(name):
        return re.sub(r'[^A-Za-z0-9_]+', '', name.lower().replace('/', '_'))

    df.columns = [clean_col_name(col) for col in df.columns]

    # 2. Convert Timestamps and Create Date Key
    df['ts_start'] = pd.to_datetime(df['ts_start'], unit='ms')
    df['ts_end'] = pd.to_datetime(df['ts_end'], unit='ms')
    df['date'] = df['ts_start'].dt.date
    df['deviceid'] = df['deviceid'].astype('category')

    # 3. Define Column Groups for Imputation
    ACTIVITY_COLS = ['steps', 'distance', 'calories']
    PHYSIO_COLS = ['hr', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50', 'lf', 'hf', 'lf_hf', 'missingness_score', 'light_avg', 'acc_x_avg', 'acc_y_avg', 'grv_w_avg','gyr_z_avg','gyr_y_avg', 'grv_x_avg','gyr_x_avg','acc_z_avg', 'grv_y_avg', 'grv_z_avg']

    # 4. CRITICAL Imputation: Two-Stage Process

    # Stage 1: Impute Activity Metrics with ZERO
    activity_cols_present = [col for col in ACTIVITY_COLS if col in df.columns]
    df[activity_cols_present] = df[activity_cols_present].fillna(0)

    # Stage 2: Impute Physiological & Context Metrics with FORWARD-FILL then MEyA
    for col in PHYSIO_COLS:
        if col in df.columns:
            df[col] = df[col].ffill() # Forward-Fill

    imputer = SimpleImputer(strategy='mean')
    for col in PHYSIO_COLS:
        if col in df.columns and df[col].isnull().any():
            df[col] = imputer.fit_transform(df[[col]]) # Mean-Fill remaining NaNs

    print(f"✅ Data Cleaned. Remaining NaNs: {df.isnull().sum().sum()}")

except Exception as e:
    print(f"❌ Error during ingestion or setup: {e}")
    df = None

--- Part 1: Data Ingestion & Initial Setup ---
✅ Data Cleaned. Remaining NaNs: 0


In [ ]:
df.isnull().sum()

,0
deviceid,0
ts_start,0
ts_end,0
missingness_score,0
hr,0
ibi,0
acc_x_avg,0
acc_y_avg,0
acc_z_avg,0
grv_x_avg,0


In [ ]:
df

,deviceid,ts_start,ts_end,missingness_score,hr,ibi,acc_x_avg,acc_y_avg,acc_z_avg,grv_x_avg,...,light_avg,sdnn,sdsd,rmssd,pnn20,pnn50,lf,hf,lf_hf,date
0,ab60,2021-04-01 07:33:45.031,2021-04-01 07:38:44.833,0.295448,84.592816,728.534374,0.284765,-0.593973,9.195984,-0.094203,...,841.324415,89.363225,78.033038,98.417649,0.746269,0.391791,1818.065625,1252.992443,1.450979,2021-04-01
1,ab60,2021-03-26 05:33:37.151,2021-03-26 05:38:36.986,0.239085,78.589565,781.896913,3.050179,-1.239353,5.790543,-0.211973,...,486.485050,100.804974,73.468687,106.602933,0.803704,0.540741,1918.499169,2784.393293,0.689019,2021-03-26
2,ab60,2021-03-26 05:28:37.083,2021-03-26 05:33:36.952,0.100773,75.620524,812.183910,2.153267,-3.546833,8.499866,-0.628970,...,779.033113,89.073518,48.675496,74.504896,0.809816,0.484663,1292.696555,1034.550027,1.249525,2021-03-26
3,ab60,2021-03-26 05:23:37.077,2021-03-26 05:28:36.883,0.268178,85.813165,769.754943,2.898409,-3.401356,4.606113,-0.249247,...,389.408638,102.371770,48.631219,70.615187,0.719178,0.400685,1561.785747,812.118619,1.923101,2021-03-26
4,ab60,2021-03-26 04:53:36.800,2021-03-26 04:58:36.672,0.043466,76.944500,775.190053,-0.050221,-6.576164,5.377019,0.715893,...,276.345515,84.045665,42.104491,66.973170,0.736111,0.450000,3460.217895,1886.157661,1.834533,2021-03-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38908,zk41,2021-03-14 16:05:51.497,2021-03-14 16:10:51.303,0.246220,65.536397,955.357204,4.807801,-1.653819,-1.580354,0.169827,...,61.632107,114.902785,71.233787,103.785382,0.814815,0.569444,2398.976663,2287.649226,1.048665,2021-03-14
38909,zk41,2021-03-14 16:10:51.502,2021-03-14 16:15:51.309,0.339678,57.850605,943.549525,7.178785,-2.219444,2.776768,-0.280727,...,102.795987,170.918050,108.960638,142.977787,0.835616,0.575342,5977.876931,4930.029492,1.212544,2021-03-14
38910,zk41,2021-03-14 16:25:51.640,2021-03-14 16:30:51.473,0.037545,55.898726,1078.602052,7.426670,-0.513623,5.921239,-0.843317,...,176.481605,101.709059,68.856070,106.702923,0.821970,0.628788,695.458077,2063.201431,0.337077,2021-03-14
38911,zk41,2021-03-15 05:39:56.304,2021-03-15 05:44:56.197,0.277592,75.857392,832.140507,0.331235,-2.814619,7.396340,-0.054537,...,45.383051,85.282439,64.734356,88.297223,0.769874,0.439331,1100.446412,2563.999375,0.429191,2021-03-15


In [ ]:
# Cell 2: Deep Learning Feature Extractor (LSTM)

if df is not None:
    print("\n--- Part 2: Deep Learning Feature Extraction (LSTM) ---")

    # --- 1. Define the LSTM Feature Extractor Model ---
    class TimeSeriesFeatureExtractor(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim):
            super(TimeSeriesFeatureExtractor, self).__init__()
            self.lstm = nn.LSTM(input_dim, hidden_dim, batch_first=True, bidirectional=True)
            self.fc = nn.Linear(hidden_dim * 2, output_dim)

        def forward(self, x):
            lstm_out, (hn, cn) = self.lstm(x)
            final_state = torch.cat((hn[-2,:,:], hn[-1,:,:]), dim=1)
            embedding = self.fc(final_state)
            return embedding

    # --- 2. Data Preparation and Extraction Setup ---
    LSTM_FEATURES = ['hr', 'rmssd', 'lf_hf', 'steps', 'light_avg', 'acc_x_avg', 'acc_y_avg', 'acc_z_avg']
    INPUT_DIM = len(LSTM_FEATURES)
    OUTPUT_EMBEDDING_DIM = 64 # Size of the new, rich feature vector
    HIDDEN_DIM = 128
    BATCH_SIZE = 1024

    # A. Scale the raw time-series data *only* for the LSTM features
    lstm_scaler = StandardScaler()
    df_lstm_input = df[LSTM_FEATURES].copy()
    df_lstm_input.iloc[:, :] = lstm_scaler.fit_transform(df_lstm_input)

    # B. Prepare data for LSTM: Reshape to (Total_Samples, Sequence_Length=1, Input_Features)
    sequence_tensor = torch.tensor(df_lstm_input.values, dtype=torch.float32).unsqueeze(1)

    # C. Setup Model and Run Extraction
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    LSTM_model = TimeSeriesFeatureExtractor(INPUT_DIM, HIDDEN_DIM, OUTPUT_EMBEDDING_DIM).to(device)
    LSTM_model.eval() # Set to evaluation mode for feature extraction

    dataloader = DataLoader(TensorDataset(sequence_tensor.to(device)), batch_size=BATCH_SIZE, shuffle=False)

    deep_embeddings = []
    with torch.no_grad():
        for [batch] in dataloader:
            embeddings = LSTM_model(batch)
            deep_embeddings.append(embeddings.cpu())

    # 3. Concatenate and Attach Deep Features to the main DataFrame
    deep_embeddings_tensor = torch.cat(deep_embeddings, dim=0)
    deep_features_df = pd.DataFrame(
        deep_embeddings_tensor.numpy(),
        index=df.index,
        columns=[f'lstm_embed_{i}' for i in range(OUTPUT_EMBEDDING_DIM)]
    )
    # CRITICAL: Attach the 64 new features back to the original DataFrame
    df = pd.concat([df, deep_features_df], axis=1)

    print(f"✅ Deep Feature Extraction Complete. Added {OUTPUT_EMBEDDING_DIM} features.")


--- Part 2: Deep Learning Feature Extraction (LSTM) ---
✅ Deep Feature Extraction Complete. Added 64 features.


In [ ]:
# Cell 3: Feature Aggregation and Change-Based Conceptual Labeling

if df is not None:
    print("\n--- Part 3: Aggregation, Fusion, and CHANGE-BASED Labeling ---")

    # --- Helper ---
    def mean_if_sleeping(series, df_ref):
        sleeping_values = series[df_ref.loc[series.index, 'is_sleeping'] == 1]
        return sleeping_values.mean() if not sleeping_values.empty else np.nan

    # 1. Rule-Based Sleep Derivation (unchanged)
    df['is_sleeping'] = np.where(
        (df['steps'] <= 5) & (df['light_avg'] < 10),
        1,
        0
    )

    # 2. Aggregate Daily Features (unchanged)
    daily_features = df.groupby(['deviceid', 'date'], observed=True).agg(
        daily_steps=('steps', 'sum'),
        daily_calories=('calories', 'sum'),
        hr_avg_24h=('hr', 'mean'),
        rmssd_24h=('rmssd', 'mean'),
        lfhf_ratio_avg=('lf_hf', 'mean'),
        total_sleep_duration_hrs=('is_sleeping', lambda x: x.sum() * 300 / 600),
        sleep_rmssd_avg=('rmssd', lambda x: mean_if_sleeping(x, df)),
        **{f'lstm_avg_{i}': (f'lstm_embed_{i}', 'mean') for i in range(OUTPUT_EMBEDDING_DIM)}
    ).reset_index()

    # -----------------------------
    # ὓ0 CHANGE-BASED FEATURES
    # -----------------------------

    # Sort for time operations
    daily_features = daily_features.sort_values(['deviceid', 'date'])

    # 3. Personal Baselines (Median per person)
    BASELINE_COLS = [
        'rmssd_24h',
        'daily_steps',
        'total_sleep_duration_hrs',
        'lfhf_ratio_avg'
    ]

    baseline = daily_features.groupby('deviceid')[BASELINE_COLS].transform('median')

    # 4. Deviation from Baseline (Δ features)
    for col in BASELINE_COLS:
        daily_features[f'delta_{col}'] = daily_features[col] - baseline[col]

    # 5. Rolling Trends (14-day slope)
    def rolling_slope(series, window=14):
        x = np.arange(window)
        return series.rolling(window).apply(
            lambda y: np.polyfit(x, y, 1)[0] if len(y) == window else np.nan,
            raw=True
        )

    daily_features['rmssd_trend_14d'] = (
        daily_features.groupby('deviceid')['rmssd_24h']
        .apply(lambda s: rolling_slope(s, 14))
        .reset_index(level=0, drop=True) # Added to fix the TypeError
    )

    daily_features['steps_trend_14d'] = (
        daily_features.groupby('deviceid')['daily_steps']
        .apply(lambda s: rolling_slope(s, 14))
        .reset_index(level=0, drop=True) # Added to fix the TypeError
    )

    daily_features['sleep_trend_14d'] = (
        daily_features.groupby('deviceid')['total_sleep_duration_hrs']
        .apply(lambda s: rolling_slope(s, 14))
        .reset_index(level=0, drop=True) # Added to fix the TypeError
    )

    # 6. Sleep Variability (7-day rolling STD)
    daily_features['sleep_variability_7d'] = (
        daily_features.groupby('deviceid')['total_sleep_duration_hrs']
        .rolling(7)
        .std()
        .reset_index(level=0, drop=True)
    )

    # -----------------------------
    # ὓ0 CHANGE-BASED RISK SCORE
    # -----------------------------

    risk_components = [
        'delta_rmssd_24h',
        'delta_daily_steps',
        'delta_total_sleep_duration_hrs',
        'rmssd_trend_14d',
        'steps_trend_14d',
        'sleep_trend_14d',
        'sleep_variability_7d'
    ]

    # Normalize components (z-score per person)
    for col in risk_components:
        daily_features[col + '_z'] = (
            daily_features.groupby('deviceid')[col]
            .transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
        )

    daily_features['Mental_Health_Risk_Score'] = daily_features[
        [c + '_z' for c in risk_components]
    ].mean(axis=1)

    # -----------------------------
    # ὓ0 FINAL LABEL (DATA-DRIVEN)
    # -----------------------------

    # High risk = top 25% risk within each individual
    daily_features['Mental_Health_Label'] = (
        daily_features.groupby('deviceid')['Mental_Health_Risk_Score']
        .transform(lambda x: (x > x.quantile(0.75)).astype(int))
    )

    print("✅ Change-based mental health risk labeling complete.")
    print("✅ No absolute physiological thresholds used.")
    print("✅ Labels are baseline-relative and longitudinal.")


--- Part 3: Aggregation, Fusion, and CHANGE-BASED Labeling ---


/tmp/ipython-input-3490304332.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  baseline = daily_features.groupby('deviceid')[BASELINE_COLS].transform('median')
/tmp/ipython-input-3490304332.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  daily_features.groupby('deviceid')['rmssd_24h']
/tmp/ipython-input-3490304332.py:66: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  daily_features.groupby('deviceid')['daily_step

✅ Change-based mental health risk labeling complete.
✅ No absolute physiological thresholds used.
✅ Labels are baseline-relative and longitudinal.


In [ ]:
daily_features

,deviceid,date,daily_steps,daily_calories,hr_avg_24h,rmssd_24h,lfhf_ratio_avg,total_sleep_duration_hrs,sleep_rmssd_avg,lstm_avg_0,...,sleep_variability_7d,delta_rmssd_24h_z,delta_daily_steps_z,delta_total_sleep_duration_hrs_z,rmssd_trend_14d_z,steps_trend_14d_z,sleep_trend_14d_z,sleep_variability_7d_z,Mental_Health_Risk_Score,Mental_Health_Label
0,ab60,2021-03-04,0.000000,0.000000,83.895188,80.442262,1.086502,0.0,NaN,0.052964,...,NaN,0.472090,-0.371408,-0.519146,NaN,NaN,NaN,NaN,-0.139488,0
1,ab60,2021-03-08,0.000000,0.000000,82.924971,76.687413,1.633419,0.5,82.136067,0.061535,...,NaN,-0.052448,-0.371408,-0.268524,NaN,NaN,NaN,NaN,-0.230793,0
2,ab60,2021-03-09,19.000000,0.786662,89.466513,53.953852,2.074060,0.0,NaN,0.065257,...,NaN,-3.228242,-0.180444,-0.519146,NaN,NaN,NaN,NaN,-1.309277,0
3,ab60,2021-03-10,3.000000,0.117493,89.308413,71.873929,1.531379,1.0,53.720814,0.058030,...,NaN,-0.724874,-0.341256,-0.017902,NaN,NaN,NaN,NaN,-0.361344,0
4,ab60,2021-03-11,55.690476,1.909073,90.699344,80.859574,1.151620,0.0,NaN,0.050016,...,NaN,0.530387,0.188325,-0.519146,NaN,NaN,NaN,NaN,0.066522,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1324,zk41,2021-04-01,0.000000,0.000000,60.617678,102.975066,0.581748,2.0,102.975066,0.066385,...,1.511858,-0.015155,-0.433363,-0.649743,0.362875,-0.522072,0.014326,-1.236793,-0.354275,0
1325,zk41,2021-04-02,0.000000,0.000000,64.312558,94.324118,0.894548,9.0,93.670599,0.066896,...,2.760262,-0.469044,-0.433363,0.451615,-0.109232,-0.960064,0.305554,-0.573897,-0.255490,0
1326,zk41,2021-04-03,21.171429,0.760575,70.927275,87.577970,0.691784,8.5,83.771229,0.064479,...,3.338092,-0.822994,0.855242,0.372947,-0.707299,-0.898686,1.061100,-0.267073,-0.058109,0
1327,zk41,2021-04-04,17.733333,0.602673,74.447118,74.988562,0.897819,6.0,74.595703,0.064522,...,3.078342,-1.483522,0.645982,-0.020396,-1.238675,-0.995229,1.725980,-0.404999,-0.252980,0


In [ ]:
# Get the value counts of the 'Mental_Health_Label' column
mental_health_label_counts = daily_features['Mental_Health_Label'].value_counts()

# Display the counts
print("Total number of mental health values for each label:")
display(mental_health_label_counts)

Total number of mental health values for each label:


,count
Mental_Health_Label,
0,992
1,337


In [ ]:
# Cell 4: Final Preparation, Scaling, and Output (Final Module 2 Output)

if 'daily_features' in locals():
    print("\n--- Part 4: Final Scaling, Splitting, and Output ---")

    # 1. Identify final feature columns (NOW INCLUDES THE 64 LSTM FEATURES)
    TRADITIONAL_FEATURES = [
        'daily_steps', 'daily_calories', 'hr_avg_24h', 'lfhf_ratio_avg',
        'rmssd_24h', 'total_sleep_duration_hrs', 'sleep_rmssd_avg'
    ]
    LSTM_AVG_FEATURES = [col for col in daily_features.columns if col.startswith('lstm_avg_')]

    FEATURE_COLUMNS_FINAL = TRADITIONAL_FEATURES + LSTM_AVG_FEATURES

    # 2. Final Data Cleanup (Dropping NaNs)
    final_data = daily_features.dropna(subset=FEATURE_COLUMNS_FINAL)

    X_wearable_full = final_data[FEATURE_COLUMNS_FINAL]
    y_wearable_labels = final_data['Mental_Health_Label']

    # 3. Data Splitting (80% Train/Fusion, 20% Test)
    X_wearable, X_wearable_test, y_wearable, y_wearable_test = train_test_split(
        X_wearable_full, y_wearable_labels, test_size=0.2, random_state=42, stratify=y_wearable_labels.values
    )

    # 4. Feature Scaling (Fit only on the training set)
    scaler = StandardScaler()
    X_wearable_scaled = scaler.fit_transform(X_wearable)

    # --- Module 2 Final Output ---
    # Convert to PyTorch tensors
    wearable_features_tensor = torch.tensor(X_wearable_scaled, dtype=torch.float32)
    wearable_labels_tensor = torch.tensor(y_wearable.values, dtype=torch.long)

    print("\n--- Module 2 Output Summary ---")
    print(f"✅ FINAL FEATURE COUNT: {wearable_features_tensor.shape[1]} (Traditional + {OUTPUT_EMBEDDING_DIM} LSTM)")
    print(f"Output Feature Shape (X_wearable): {wearable_features_tensor.shape}")


--- Part 4: Final Scaling, Splitting, and Output ---

--- Module 2 Output Summary ---
✅ FINAL FEATURE COUNT: 71 (Traditional + 64 LSTM)
Output Feature Shape (X_wearable): torch.Size([892, 71])


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import re # Import re for regular expressions

# --- Variables from the code you provided: ---
X_tsne_wearable = X_wearable_scaled
y_tsne_wearable = y_wearable.values

# 1. Apply t-SNE Transformation
print("Applying t-SNE to reduce wearable features to 2D...")
# Note: The input feature dimension is 71 (7 traditional + 64 LSTM_AVG_FEATURES)
tsne = TSNE(n_components=2, random_state=42, verbose=1, n_jobs=-1, perplexity=30)
X_tsne_2d_wearable = tsne.fit_transform(X_tsne_wearable)

# 2. Plotting the t-SNE Results
tsne_df_wearable = pd.DataFrame(data=X_tsne_2d_wearable, columns=['tsne_1', 'tsne_2'])
tsne_df_wearable['label'] = y_tsne_wearable

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    tsne_df_wearable['tsne_1'],
    tsne_df_wearable['tsne_2'],
    c=tsne_df_wearable['label'],
    cmap='RdYlBu', # Uses a divergent color map for clear separation
    alpha=0.6,
    s=10
)
# Labels are now inverted: original 0 (Low Risk) is now 1, original 1 (High Risk) is now 0
legend_labels = {0: 'High Risk (0)', 1: 'Low Risk (1)'}
legend_handles = scatter.legend_elements()[0]

# --- FIX for the KeyError ---
# Extract the integer labels from the LaTeX formatted strings for legend lookup
legend_keys = [int(re.search(r'\d+', key).group()) for key in scatter.legend_elements()[1]]

plt.legend(legend_handles, [legend_labels[key] for key in legend_keys],
           loc="lower left", title="Depression Risk")

plt.title('t-SNE Visualization of Aggregated Wearable Features (Module 2 - Inverted Labels)')
plt.xlabel('t-SNE Component 1')
plt.ylabel('t-SNE Component 2')
plt.savefig('tsne_visualization_module2_inverted.png')
plt.show()
print("t-SNE visualization saved as 'tsne_visualization_module2_inverted.png'.")

Applying t-SNE to reduce wearable features to 2D...
[t-SNE] Computing 91 nearest neighbors...
[t-SNE] Indexed 892 samples in 0.001s...
[t-SNE] Computed neighbors for 892 samples in 0.271s...
[t-SNE] Computed conditional probabilities for sample 892 / 892
[t-SNE] Mean sigma: 2.442727
[t-SNE] KL divergence after 250 iterations with early exaggeration: 68.145195


KeyboardInterrupt: 

In [ ]:
import os
import torch

# Define the base path in Google Drive
save_path = '/content/drive/MyDrive/depjct/depjct_outputs_2'

# Create the directory if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Created directory: {save_path}")

print(f"Saving outputs to: {save_path}")

# 1. Save the trained Transformer model
model_save_filepath = os.path.join(save_path, 'twhin_bert_sentiment_model')
model.save_pretrained(model_save_filepath)
print(f"✅ Trained model saved to {model_save_filepath}")

# 2. Save the tokenizer
tokenizer_save_filepath = os.path.join(save_path, 'twhin_bert_tokenizer')
tokenizer.save_pretrained(tokenizer_save_filepath)
print(f"✅ Tokenizer saved to {tokenizer_save_filepath}")

# 3. Save Module 1 feature vectors (if they exist)
if 'feature_vectors' in locals() and feature_vectors is not None:
    # Assuming feature_vectors might be a list of tensors, concatenate if necessary
    if isinstance(feature_vectors, list):
        feature_vectors_tensor = torch.cat(feature_vectors, dim=0)
    else:
        feature_vectors_tensor = feature_vectors
    torch.save(feature_vectors_tensor, os.path.join(save_path, 'module1_feature_vectors.pt'))
    print(f"✅ Module 1 feature vectors saved to {os.path.join(save_path, 'module1_feature_vectors.pt')}")

# 3.5. Save Module 1 labels (y_val_tensor) as part of module 1 outputs
if 'y_val_tensor' in locals() and y_val_tensor is not None:
    torch.save(y_val_tensor, os.path.join(save_path, 'module1_labels.pt'))
    print(f"✅ Module 1 labels saved to {os.path.join(save_path, 'module1_labels.pt')}")

# 4. Save Module 2 wearable features and labels
if 'wearable_features_tensor' in locals() and wearable_features_tensor is not None:
    torch.save(wearable_features_tensor, os.path.join(save_path, 'module2_wearable_features.pt'))
    print(f"✅ Module 2 wearable features saved to {os.path.join(save_path, 'module2_wearable_features.pt')}")

if 'wearable_labels_tensor' in locals() and wearable_labels_tensor is not None:
    torch.save(wearable_labels_tensor, os.path.join(save_path, 'module2_wearable_labels.pt'))
    print(f"✅ Module 2 wearable labels saved to {os.path.join(save_path, 'module2_wearable_labels.pt')}")


print("All requested outputs successfully saved to Google Drive!")
print("Note: The visualization plots (t-SNE and PCA) were saved to the root of your Colab content directory earlier by their respective cells. You might want to move them to the 'depjct_outputs' folder manually if desired.")

In [ ]:
import os
import torch

# Define the base path in Google Drive
save_path = '/content/drive/MyDrive/depjct/depjct_outputs_2'

# Create the directory if it doesn't exist
if not os.path.exists(save_path):
    os.makedirs(save_path)
    print(f"Created directory: {save_path}")

print(f"Saving outputs to: {save_path}")

# 4. Save Module 2 wearable features and labels
if 'wearable_features_tensor' in locals() and wearable_features_tensor is not None:
    torch.save(wearable_features_tensor, os.path.join(save_path, 'module2_wearable_featuresold.pt'))
    print(f"✅ Module 2 wearable features saved to {os.path.join(save_path, 'module2_wearable_featuresold.pt')}")

if 'wearable_labels_tensor' in locals() and wearable_labels_tensor is not None:
    torch.save(wearable_labels_tensor, os.path.join(save_path, 'module2_wearable_labelsold.pt'))
    print(f"✅ Module 2 wearable labels saved to {os.path.join(save_path, 'module2_wearable_labelsold.pt')}")

Saving outputs to: /content/drive/MyDrive/depjct/depjct_outputs_2
✅ Module 2 wearable features saved to /content/drive/MyDrive/depjct/depjct_outputs_2/module2_wearable_featuresold.pt
✅ Module 2 wearable labels saved to /content/drive/MyDrive/depjct/depjct_outputs_2/module2_wearable_labelsold.pt


In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import re

# Ensure the clean_text function is defined (as it was used previously)
def clean_text(text):
    """
    Cleans the input text by removing special characters, extra spaces, and converting to lowercase.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Define the path where the model and tokenizer were saved
save_path = '/content/drive/MyDrive/depjct/depjct_outputs_2'
model_save_filepath = os.path.join(save_path, 'twhin_bert_sentiment_model')
tokenizer_save_filepath = os.path.join(save_path, 'twhin_bert_tokenizer')

# Load the tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(tokenizer_save_filepath)
model = AutoModelForSequenceClassification.from_pretrained(model_save_filepath)

# Determine the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- Manual Input Check ---
# This section allows you to input text manually and see the model's prediction.

# Get input from the user
manual_input_text = input("Enter text to check for depression risk: ")

# Clean the input text using the same cleaning function
cleaned_input_text = clean_text(manual_input_text)

# Tokenize the cleaned input text
input_encodings = tokenizer(
    cleaned_input_text,
    max_length=128,
    padding=True,
    truncation=True,
    return_tensors='pt'  # Returns PyTorch tensors
)

# Move tensors to the appropriate device (GPU if available)
input_ids = input_encodings['input_ids'].to(device)
attention_mask = input_encodings['attention_mask'].to(device)

# Set the model to evaluation mode
model.eval()

# Make a prediction
with torch.no_grad():
    outputs = model(input_ids, attention_mask=attention_mask)

# Get the predicted label (0 or 1)
logits = outputs.logits
predicted_label = torch.argmax(logits, dim=1).item()

# Interpret the prediction (original 1 is now 0, original 0 is now 1)
if predicted_label == 0:
    print("\nBased on the input text, the model predicts a higher risk of depression.")
else:
    print("\nBased on the input text, the model predicts a lower risk of depression.")

# **MODULE 3**

In [ ]:
hi

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import os # Import os for path joining

# Define the path where the tensors were saved
save_path = '/content/drive/MyDrive/depjct/depjct_outputs_2'

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Load saved tensors ---
print(f"Loading tensors from: {save_path}")

# Load Module 1 feature vectors
social_media_features_tensor = torch.load(os.path.join(save_path, 'module1_feature_vectors.pt'))
print(f"Loaded social media features tensor shape: {social_media_features_tensor.shape}")

# Load Module 2 wearable features
wearable_features_tensor = torch.load(os.path.join(save_path, 'module2_wearable_featuresold.pt'))
print(f"Loaded wearable features tensor shape: {wearable_features_tensor.shape}")

# Load Module 1 labels (y_val_tensor)
social_media_labels_tensor = torch.load(os.path.join(save_path, 'module1_labels.pt'))
print(f"Loaded social media labels tensor shape: {social_media_labels_tensor.shape}")


# --- 1. Conceptual Alignment and Early Fusion ---
print("\n--- Step 1: Conceptual Alignment and Early Fusion ---")

# A. Determine the target size for fusion (The smaller of the two datasets)
N_social = social_media_features_tensor.size(0)
N_wearable = wearable_features_tensor.size(0)
N_min = min(N_social, N_wearable)

# B. Sample Features and Labels to align the length
# We take the first N_min samples from each dataset, ensuring labels align with social media features.
F_social = social_media_features_tensor[:N_min, :]
F_wearable = wearable_features_tensor[:N_min, :]
Y_fused = social_media_labels_tensor[:N_min]

# C. Concatenation (Early Fusion)
# Join the two feature tensors side-by-side (along dimension 1)
F_fused = torch.cat((F_social, F_wearable), dim=1)

print(f"✅ Fusion Complete. Fused Feature Shape: {F_fused.shape}")


# --- 2. Final Data Split and DataLoader Setup ---

# Split the fused data into Train/Test sets
X_train_fused, X_test_fused, y_train_fused, y_test_fused = train_test_split(
    F_fused.cpu().numpy(), Y_fused.cpu().numpy(), test_size=0.2, random_state=42, stratify=Y_fused.cpu().numpy()
)

# Convert back to PyTorch Tensors and move to device
X_train_fused = torch.tensor(X_train_fused, dtype=torch.float32).to(device)
X_test_fused = torch.tensor(X_test_fused, dtype=torch.float32).to(device)
y_train_fused = torch.tensor(y_train_fused, dtype=torch.long).to(device)
y_test_fused = torch.tensor(y_test_fused, dtype=torch.long).to(device)


# Create DataLoader for batching
BATCH_SIZE = 32
train_dataset = TensorDataset(X_train_fused, y_train_fused)
test_dataset = TensorDataset(X_test_fused, y_test_fused)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


# --- 3. Final AI Classification Model (Dense Neural Network - DNN) ---

# Define the DNN Architecture
class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim):
        super(FusedDNNClassifier, self).__init__()
        # Initial large layer handles the high-dimensional fused input (777 features)
        self.layer_1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.layer_2 = nn.Linear(128, 128)
        self.layer_3 = nn.Linear(128, 2) # Output 2 classes (0: Low Risk, 1: High Risk)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        x = self.relu(x)
        x = self.layer_3(x)
        return x

# Initialize Model, Loss, and Optimizer
INPUT_DIM = F_fused.shape[1]
model = FusedDNNClassifier(INPUT_DIM).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)
EPOCHS = 64


# --- 4. Model Training and Evaluation ---

# Training Loop
model.train()
print("\n--- Training Final Multimodal Classifier ---")
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in train_dataloader:
        inputs, labels = [t.to(device) for t in batch]

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_fn(outputs, labels)
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{EPOCHS} Training Loss: {total_loss/len(train_dataloader):.4f}")


# Final Evaluation
model.eval()
all_predictions = []
all_true_labels = []

with torch.no_grad():
    for batch in test_dataloader:
        inputs, labels = [t.to(device) for t in batch]
        outputs = model(inputs)
        _, preds = torch.max(outputs, dim=1) # Get the predicted class index
        all_predictions.extend(preds.cpu().numpy())
        all_true_labels.extend(labels.cpu().numpy())

# Calculate Final Metrics
f1 = f1_score(all_true_labels, all_predictions, average='binary')
print("\n--- Final Module 3 Evaluation ---")
print(f"✅ Fused Model Test F1-Score: {f1:.4f}")
print("\n--- Final Classification Report ---")
print(classification_report(all_true_labels, all_predictions, target_names=['Low Risk (0)', 'High Risk (1)']))

# Save the trained FusedDNNClassifier model
multimodal_model_save_filepath = os.path.join(save_path, 'fused_dnn_classifier_model.pt')
torch.save(model.state_dict(), multimodal_model_save_filepath)
print(f"✅ Fused DNN Classifier model saved to {multimodal_model_save_filepath}")


Loading tensors from: /content/drive/MyDrive/depjct/depjct_outputs_2
Loaded social media features tensor shape: torch.Size([10000, 768])
Loaded wearable features tensor shape: torch.Size([892, 71])
Loaded social media labels tensor shape: torch.Size([10000])

--- Step 1: Conceptual Alignment and Early Fusion ---
✅ Fusion Complete. Fused Feature Shape: torch.Size([892, 839])

--- Training Final Multimodal Classifier ---
Epoch 1/64 Training Loss: 0.2937
Epoch 2/64 Training Loss: 0.2640
Epoch 3/64 Training Loss: 0.2426
Epoch 4/64 Training Loss: 0.2419
Epoch 5/64 Training Loss: 0.2379
Epoch 6/64 Training Loss: 0.2282
Epoch 7/64 Training Loss: 0.2171
Epoch 8/64 Training Loss: 0.2051
Epoch 9/64 Training Loss: 0.2205
Epoch 10/64 Training Loss: 0.2058
Epoch 11/64 Training Loss: 0.1865
Epoch 12/64 Training Loss: 0.1876
Epoch 13/64 Training Loss: 0.2181
Epoch 14/64 Training Loss: 0.1809
Epoch 15/64 Training Loss: 0.1915
Epoch 16/64 Training Loss: 0.1758
Epoch 17/64 Training Loss: 0.1545
Epoch 18

In [ ]:
# =========================================================
# GOOGLE COLAB – Multimodal Depression Prediction (Text + Wearables)
# =========================================================

# ---------- INSTALL ----------

# ---------- MOUNT DRIVE ----------
from google.colab import drive
drive.mount('/content/drive')

# ---------- IMPORTS ----------
import os
import re
import random
import traceback
import io
from typing import Optional, Tuple

import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gradio as gr
from functools import lru_cache

# ---------- CONFIG (YOUR PATHS) ----------
SAVE_PATH  = "/content/drive/MyDrive/depjct/depjct_outputs_2"
SAVE1_PATH = "/content/drive/MyDrive/depjct_outputs"

TWHIN_TOKENIZER_DIR = os.path.join(SAVE1_PATH, "twhin_bert_tokenizer")
TWHIN_MODEL_DIR     = os.path.join(SAVE1_PATH, "twhin_bert_sentiment_model")
WEARABLE_PATH       = os.path.join(SAVE_PATH,  "module2_wearable_features.pt")
FUSED_MODEL_PATH    = os.path.join(SAVE_PATH,  "fused_dnn_classifier_model.pt")

LOCAL_ONLY = True

TRADITIONAL_FEATURES = [
    "daily_steps",
    "daily_calories",
    "hr_avg_24h",
    "lfhf_ratio_avg",
    "rmssd_24h",
    "total_sleep_duration_hrs",
    "sleep_efficiency"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------- PREFLIGHT CHECK ----------
def preflight_check():
    missing = []
    if not os.path.isdir(TWHIN_TOKENIZER_DIR):
        missing.append("Tokenizer folder missing")
    if not os.path.isdir(TWHIN_MODEL_DIR):
        missing.append("TWHIN model folder missing")
    if not os.path.isfile(WEARABLE_PATH):
        missing.append("Wearable tensor missing")
    if not os.path.isfile(FUSED_MODEL_PATH):
        missing.append("Fused model missing")
    return missing

missing = preflight_check()
if missing:
    raise RuntimeError("Preflight failed: " + ", ".join(missing))
else:
    print("✅ Preflight OK")

# ---------- UTILITIES ----------
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

# ---------- FUSED MODEL ----------
class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.drop = nn.Dropout(0.25)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.drop(x)
        x = self.relu(self.fc2(x))
        return self.fc3(x)

# ---------- LOADERS ----------
@lru_cache(maxsize=1)
def load_twhin():
    tokenizer = AutoTokenizer.from_pretrained(
        TWHIN_TOKENIZER_DIR, local_files_only=LOCAL_ONLY
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        TWHIN_MODEL_DIR,
        output_hidden_states=True,
        local_files_only=LOCAL_ONLY
    ).to(device).eval()
    return tokenizer, model

@lru_cache(maxsize=1)
def load_wearable():
    w = torch.load(WEARABLE_PATH, map_location="cpu")
    if w.ndim == 1:
        w = w.unsqueeze(0)
    return w

@lru_cache(maxsize=1)
def load_fused():
    _, twhin = load_twhin()
    wearable = load_wearable()
    input_dim = twhin.config.hidden_size + wearable.shape[1]
    model = FusedDNNClassifier(input_dim)
    model.load_state_dict(torch.load(FUSED_MODEL_PATH, map_location=device))
    return model.to(device).eval()

# ---------- FORMAT HELPERS ----------
def wearable_scaled_df(t: torch.Tensor):
    return pd.DataFrame([t.flatten().tolist()])

def wearable_named_df(t: torch.Tensor):
    vals = t.flatten().tolist()
    cols = TRADITIONAL_FEATURES[:len(vals)]
    return pd.DataFrame([vals], columns=cols)

# ---------- MAIN PREDICTION ----------
def fused_predict(text):
    try:
        tokenizer, twhin = load_twhin()
        wearable_tensor = load_wearable()
        fused = load_fused()

        text = clean_text(text)
        enc = tokenizer(
            text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=128
        ).to(device)

        with torch.no_grad():
            out = twhin(**enc)
            text_feat = out.hidden_states[-1][:, 0, :]

        idx = random.randint(0, wearable_tensor.shape[0] - 1)
        wear = wearable_tensor[idx].to(device)

        fused_input = torch.cat([text_feat, wear.unsqueeze(0)], dim=1)

        with torch.no_grad():
            logits = fused(fused_input)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]

        label = "HIGH RISK" if probs[0] > probs[1] else "LOW RISK"

        md = f"""
### 🧠 Multimodal Depression Prediction

**Prediction:** **{label}**

- High Risk Probability: `{probs[0]:.4f}`
- Low Risk Probability : `{probs[1]:.4f}`

_Wearable sample index: {idx}_
"""

        return md, wearable_named_df(wear.cpu())

    except Exception as e:
        return f"❌ Error: {e}", pd.DataFrame()

# ---------- GRADIO UI ----------
iface = gr.Interface(
    fn=fused_predict,
    inputs=gr.Textbox(lines=5, label="Enter social media text"),
    outputs=[
        gr.Markdown(label="Prediction"),
        gr.Dataframe(label="Wearable Features (Unscaled)")
    ],
    title="DEPRESSION PREDICTION MODEL",
    description="Multimodal fusion of text + wearable signals"
)

iface.launch(debug=True, share=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
✅ Preflight OK
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cb5be55879d5c3dd88.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.


KeyboardInterrupt: 

In [ ]:
# =========================================================
# SINGLE-CELL COLAB SCRIPT (FIXED & WORKING)
# Multimodal Mental Health Risk Monitoring (Text + Wearables)
# =========================================================

# ---------- INSTALL & MOUNT ----------
from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers gradio torch

# ---------- IMPORTS ----------
import os, re, traceback
from typing import Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gradio as gr

# ---------- CONFIG (EDIT PATHS IF NEEDED) ----------
SAVE_PATH  = "/content/drive/MyDrive/depjct/depjct_outputs_2"
SAVE1_PATH = "/content/drive/MyDrive/depjct_outputs"

TWHIN_TOKENIZER_DIR = os.path.join(SAVE1_PATH, "twhin_bert_tokenizer")
TWHIN_MODEL_DIR     = os.path.join(SAVE1_PATH, "twhin_bert_sentiment_model")
WEARABLE_PATH       = os.path.join(SAVE_PATH, "module2_wearable_features.pt")
FUSED_MODEL_PATH    = os.path.join(SAVE_PATH, "fused_dnn_classifier_model.pt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------- PREFLIGHT CHECK ----------
def preflight_check() -> Tuple[bool, str]:
    missing = []
    if not os.path.isdir(TWHIN_TOKENIZER_DIR):
        missing.append("Tokenizer missing")
    if not os.path.isdir(TWHIN_MODEL_DIR):
        missing.append("Text model missing")
    if not os.path.isfile(WEARABLE_PATH):
        missing.append("Wearable tensor missing")
    if not os.path.isfile(FUSED_MODEL_PATH):
        missing.append("Fused model missing")
    return len(missing) == 0, ", ".join(missing)

ok, msg = preflight_check()
if not ok:
    raise RuntimeError("❌ Preflight failed: " + msg)
print("✅ Preflight OK")

# ---------- UTIL ----------
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

# ---------- FUSED MODEL ----------
class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim: int):
        super(FusedDNNClassifier, self).__init__()
        self.layer_1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.layer_2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.layer_3 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        x = self.relu2(x)
        x = self.layer_3(x)
        return x

# ---------- LOAD MODELS (NO CACHE → COLAB SAFE) ----------
def load_text_model():
    tokenizer = AutoTokenizer.from_pretrained(
        TWHIN_TOKENIZER_DIR,
        local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        TWHIN_MODEL_DIR,
        output_hidden_states=True,
        local_files_only=True
    ).to(device)
    model.eval()
    return tokenizer, model

def load_wearable_tensor():
    w = torch.load(WEARABLE_PATH, map_location="cpu")
    if w.ndim != 2:
        raise ValueError("Wearable tensor must be [days, features]")
    return w

def load_fused_model():
    tokenizer, text_model = load_text_model()
    wearable = load_wearable_tensor()

    input_dim = text_model.config.hidden_size + wearable.shape[1]
    fused = FusedDNNClassifier(input_dim)

    fused.load_state_dict(
        torch.load(FUSED_MODEL_PATH, map_location=device)
    )
    fused.to(device)
    fused.eval()
    return fused

# ---------- PREDICTION FUNCTION ----------
def fused_predict(text):
    print("SUBMIT CLICKED")  # DEBUG CONFIRMATION

    if not isinstance(text, str) or not text.strip():
        return "⚠️ Please enter valid text."

    try:
        tokenizer, text_model = load_text_model()
        fused_model = load_fused_model()
        wearable = load_wearable_tensor()

        enc = tokenizer(
            clean_text(text),
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        with torch.no_grad():
            out = text_model(
                enc["input_ids"].to(device),
                attention_mask=enc["attention_mask"].to(device)
            )
            text_feat = out.hidden_states[-1][:, 0, :].squeeze(0)

        wearable_feat = wearable.mean(dim=0)

        fused_input = torch.cat(
            [text_feat.cpu(), wearable_feat],
            dim=0
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            probs = F.softmax(
                fused_model(fused_input),
                dim=1
            )[0].cpu().numpy()

        label = (
            "LOW MENTAL HEALTH RISK"
            if probs[0] > probs[1]
            else "HIGH MENTAL HEALTH RISK"
        )

        return f"""
### 🧠 Multimodal Mental Health Risk Assessment

**Risk State:** **{label}**

- Low Risk: **{probs[0]:.4f}**
- High Risk: **{probs[1]:.4f}**

⚠️ _Not a clinical diagnosis_
"""

    except Exception:
        return f"❌ Error:\n```\n{traceback.format_exc()}\n```"

# ---------- GRADIO UI (EXPLICIT BUTTON) ----------
with gr.Blocks() as iface:
    gr.Markdown("# 🧠 MULTIMODAL MENTAL HEALTH RISK MONITORING")

    txt = gr.Textbox(
        lines=5,
        label="Enter social media / diary text"
    )

    btn = gr.Button("Submit")
    out = gr.Markdown()

    btn.click(
        fn=fused_predict,
        inputs=txt,
        outputs=out
    )

iface.queue()

iface.launch(share=True, debug=True)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
✅ Preflight OK
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0fcf0bf941b9cb59ba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


SUBMIT CLICKED
SUBMIT CLICKED
SUBMIT CLICKED
SUBMIT CLICKED


In [ ]:
import os
import torch
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import random # For selecting a random wearable feature vector
import torch.nn as nn # Need to redefine the FusedDNNClassifier class
import pandas as pd # Import pandas to handle DataFrame operations
import torch.nn.functional as F # For softmax

# Define the clean_text function (must be available or defined here)
def clean_text(text):
    """
    Cleans the input text by removing special characters, extra spaces, and converting to lowercase.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Define the FusedDNNClassifier class (must be available to load state_dict)
class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim):
        super(FusedDNNClassifier, self).__init__()
        self.layer_1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.layer_2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.layer_3 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        x = self.relu2(x)
        x = self.layer_3(x)
        return x

# Define the path where models and tensors were saved
save_path = '/content/drive/MyDrive/depjct/depjct_outputs_2'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("--- Setting up for Manual Multimodal Prediction ---")

# 1. Load TwHIN-BERT Tokenizer and Model (for text feature extraction and individual prediction)
twhin_tokenizer_path = os.path.join(save_path, 'twhin_bert_tokenizer')
twhin_model_path = os.path.join(save_path, 'twhin_bert_sentiment_model')

# --- BEGIN FIX FOR OSERROR ---
# Check if the model weight file exists before trying to load
model_weight_file = os.path.join(twhin_model_path, 'pytorch_model.bin') # Common weight file name
if not os.path.exists(model_weight_file):
    # Fallback to .safetensors if .bin not found, or raise an informative error
    model_weight_file = os.path.join(twhin_model_path, 'model.safetensors')
    if not os.path.exists(model_weight_file):
        raise FileNotFoundError(
            f"Error: Model weight file (pytorch_model.bin or model.safetensors) not found in {twhin_model_path}. "
            "Please ensure cell `kcuDf5F1EX8X` (saving the model) was run successfully and completely before executing this cell."
        )


twhin_tokenizer = AutoTokenizer.from_pretrained(twhin_tokenizer_path, fix_mistral_regex=True)
twhin_model = AutoModelForSequenceClassification.from_pretrained(twhin_model_path, output_hidden_states=True)
# --- END FIX FOR OSERROR ---

twhin_model.to(device)
twhin_model.eval() # Set to evaluation mode for feature extraction and prediction
print("✅ TwHIN-BERT model and tokenizer loaded for text feature extraction.")

# 2. Load Wearable Features Tensor (to sample from)
wearable_features_tensor = torch.load(os.path.join(save_path, 'module2_wearable_features.pt'))
print(f"✅ Wearable features tensor loaded (shape: {wearable_features_tensor.shape}).")

# 3. Load the trained FusedDNNClassifier model
# Determine the input dimension based on the feature shapes
# (TwHIN-BERT hidden size is 768 for bert-base, wearable features is 71)
FUSED_DNN_INPUT_DIM = twhin_model.config.hidden_size + wearable_features_tensor.shape[1]

multimodal_model = FusedDNNClassifier(FUSED_DNN_INPUT_DIM)
multimodal_model_save_filepath = os.path.join(save_path, 'fused_dnn_classifier_model.pt')
multimodal_model.load_state_dict(torch.load(multimodal_model_save_filepath, map_location=device))
multimodal_model.to(device)
multimodal_model.eval() # Set to evaluation mode
print("✅ Fused DNN Classifier model loaded.")


# --- Manual Multimodal Prediction ---
print("\n--- Performing Manual Multimodal Prediction ---")

# Get text input from the user
manual_input_text = input("Enter social media text: ")
cleaned_input_text = clean_text(manual_input_text)

# --- Individual Text Model Prediction ---
with torch.no_grad():
    text_encodings_for_pred = twhin_tokenizer(
        cleaned_input_text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    text_input_ids_for_pred = text_encodings_for_pred['input_ids'].to(device)
    text_attention_mask_for_pred = text_encodings_for_pred['attention_mask'].to(device)
    text_outputs_for_pred = twhin_model(text_input_ids_for_pred, attention_mask=text_attention_mask_for_pred)

    text_logits = text_outputs_for_pred.logits
    text_probabilities = F.softmax(text_logits, dim=1).cpu().squeeze(0).numpy()
    text_predicted_label = torch.argmax(text_logits, dim=1).item()

print("\n--- Individual Text Model (TwHIN-BERT) Prediction ---")
print(f"Input Text: '{manual_input_text}'")
# Interpret prediction with inverted labels: 0 is now Higher Risk, 1 is now Lower Risk
print(f"Text Model Prediction: {'Higher Risk' if text_predicted_label == 0 else 'Lower Risk'}")
# Probabilities: original 0 (Low Risk) is now 1, original 1 (High Risk) is now 0
print(f"Text Model Probabilities: High Risk={text_probabilities[0]:.4f}, Low Risk={text_probabilities[1]:.4f}")

# --- Multimodal Prediction ---
# Extract features from text (for multimodal model)
with torch.no_grad():
    text_encodings = twhin_tokenizer(
        cleaned_input_text,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    text_input_ids = text_encodings['input_ids'].to(device)
    text_attention_mask = text_encodings['attention_mask'].to(device)
    text_outputs = twhin_model(text_input_ids, attention_mask=text_attention_mask)
    # The feature vector for classification is the [CLS] token embedding of the last hidden state
    social_media_features = text_outputs.hidden_states[-1][:, 0, :].cpu().squeeze(0) # Remove batch dimension

print(f"\n✅ Text features extracted (shape: {social_media_features.shape}).")

# Simulate wearable data input (by randomly selecting one from the loaded tensor)
# In a real application, this would come from actual wearable sensor data processing.
random_idx = random.randint(0, wearable_features_tensor.shape[0] - 1)
wearable_single_feature = wearable_features_tensor[random_idx].cpu().squeeze(0) # Ensure it's a 1D tensor

# Display the corresponding unscaled traditional wearable features if X_wearable and TRADITIONAL_FEATURES are available
# Note: This block assumes X_wearable and TRADITIONAL_FEATURES are available in the global scope
# Since they are not explicitly defined here, this part might need adjustment or a new cell for context.
# For now, let's keep the logic as is, assuming those variables are brought in by previous executions.

# Check if X_wearable is defined in the global scope and TRADITIONAL_FEATURES too
if 'X_wearable' in globals() and 'TRADITIONAL_FEATURES' in globals():
    if random_idx < len(X_wearable):
        selected_unscaled_wearable_features = X_wearable.iloc[random_idx][TRADITIONAL_FEATURES]
        print("\n--- Simulated Traditional Wearable Features ---")
        display(pd.DataFrame(selected_unscaled_wearable_features).T)
    else:
        print(f"Warning: random_idx ({random_idx}) out of bounds for X_wearable (length {len(X_wearable)}). Cannot display unscaled features.")
else:
    print("Warning: X_wearable or TRADITIONAL_FEATURES not found in global scope. Cannot display unscaled features.")


print(f"   (Note: In a real scenario, these would come from actual sensor data for the individual and day.)")


# Fuse the features
fused_input = torch.cat((social_media_features, wearable_single_feature), dim=0).unsqueeze(0) # Add batch dimension
fused_input = fused_input.to(device)

print(f"✅ Fused input prepared (shape: {fused_input.shape}).")

# Make multimodal prediction
with torch.no_grad():
    multimodal_outputs = multimodal_model(fused_input)
    multimodal_logits = multimodal_outputs.cpu().squeeze(0).numpy()
    multimodal_probabilities = F.softmax(torch.tensor(multimodal_logits), dim=0).numpy()
    multimodal_predicted_label = torch.argmax(torch.tensor(multimodal_logits)).item()

print("\n--- Fused Multimodal Model Prediction ---")
# Interpret prediction with inverted labels: 0 is now Higher Risk, 1 is now Lower Risk
print(f"Multimodal Prediction: {'Higher Risk' if multimodal_predicted_label == 0 else 'Lower Risk'}")
# Probabilities: original 0 (Low Risk) is now 1, original 1 (High Risk) is now 0
print(f"Multimodal Probabilities: High Risk={multimodal_probabilities[0]:.4f}, Low Risk={multimodal_probabilities[1]:.4f}")

In [ ]:
# ============================================================
# MULTIMODAL DEPRESSION PREDICTION (COLAB)
# Text (TWHIN-BERT) + Wearables + Fused DNN
# ============================================================

# ---------------------------
# INSTALL & MOUNT
# ---------------------------
!pip install -q transformers gradio torch pandas

from google.colab import drive
drive.mount('/content/drive')

# ---------------------------
# IMPORTS
# ---------------------------
import os, re, random, io, traceback
from typing import Optional
from functools import lru_cache

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import gradio as gr

# ---------------------------
# CONFIG (EDIT ONLY THIS PATH)
# ---------------------------
SAVE_PATH = "/content/drive/MyDrive/depjct/depjct_outputs_2" # Corrected path to point to the actual output directory

# Files/folders expected inside SAVE_PATH:
# Align SAVE1_PATH with SAVE_PATH for consistency, as all outputs are in SAVE_PATH
SAVE1_PATH = SAVE_PATH

TWHIN_TOKENIZER_DIR = os.path.join(SAVE1_PATH, "twhin_bert_tokenizer")
TWHIN_MODEL_DIR     = os.path.join(SAVE1_PATH, "twhin_bert_sentiment_model")
WEARABLE_PATH       = os.path.join(SAVE_PATH, "module2_wearable_features.pt")
FUSED_MODEL_PATH    = os.path.join(SAVE_PATH, "fused_dnn_classifier_model.pt")
X_WEARABLE_CSV_PATH = os.path.join(SAVE_PATH, "module2_traditional_wearable_test_features.csv")

TRADITIONAL_FEATURES = [
    "daily_steps",
    "daily_calories",
    "hr_avg_24h",
    "lfhf_ratio_avg",
    "rmssd_24h",
    "total_sleep_duration_hrs",
    "sleep_efficiency"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------------------
# PREFLIGHT CHECK
# ---------------------------
def preflight_check():
    missing = []
    for p, name in [
        (TWHIN_TOKENIZER_DIR, "Tokenizer"),
        (TWHIN_MODEL_DIR, "Text model"),
        (WEARABLE_PATH, "Wearable tensor"),
        (FUSED_MODEL_PATH, "Fused model"),
    ]:
        if not os.path.exists(p):
            missing.append(name)
    if missing:
        raise RuntimeError("Missing files: " + ", ".join(missing))

preflight_check()
print("✅ Preflight OK")

# ---------------------------
# UTILITIES
# ---------------------------
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, 2)
        self.drop = nn.Dropout(0.25)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.drop(x)
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# ---------------------------
# LOADERS (CACHED)
# ---------------------------
@lru_cache(maxsize=1)
def load_text_model():
    tokenizer = AutoTokenizer.from_pretrained(
        TWHIN_TOKENIZER_DIR, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        TWHIN_MODEL_DIR,
        output_hidden_states=True,
        local_files_only=True
    ).to(device).eval()
    return tokenizer, model

@lru_cache(maxsize=1)
def load_wearables():
    w = torch.load(WEARABLE_PATH, map_location="cpu")
    return w.unsqueeze(0) if w.ndim == 1 else w

@lru_cache(maxsize=1)
def load_fused():
    _, txt_model = load_text_model()
    wear = load_wearables()
    input_dim = txt_model.config.hidden_size + wear.shape[1]

    model = FusedDNNClassifier(input_dim)
    model.load_state_dict(torch.load(FUSED_MODEL_PATH, map_location=device))
    model.to(device).eval()
    return model

# ---------------------------
# DATAFRAME HELPERS
# ---------------------------
def wearable_scaled_df(t):
    return pd.DataFrame([t.tolist()], columns=[f"feat_{i}" for i in range(len(t))])

def try_get_unscaled_df(idx, uploaded_df):
    def sanitize(df):
        df = df.loc[:, ~df.columns.str.contains("^Unnamed")]
        return df.drop(columns=["label"], errors="ignore")

    if uploaded_df is not None:
        df = sanitize(uploaded_df)
        cols = [c for c in TRADITIONAL_FEATURES if c in df.columns]
        if cols:
            return df.iloc[[0]][cols]

    if os.path.isfile(X_WEARABLE_CSV_PATH):
        df = sanitize(pd.read_csv(X_WEARABLE_CSV_PATH))
        cols = [c for c in TRADITIONAL_FEATURES if c in df.columns]
        if cols:
            return df.iloc[[idx % len(df)]][cols]

    return None

# ---------------------------
# MAIN PREDICTION FUNCTION
# ---------------------------
def fused_predict_show_wearable(text, wearable_csv=None):
    try:
        tokenizer, text_model = load_text_model()
        wear_tensor = load_wearables()
        fused_model = load_fused()

        uploaded_df = None
        if wearable_csv is not None:
            uploaded_df = pd.read_csv(wearable_csv.name)

        enc = tokenizer(
            clean_text(text),
            max_length=128,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            out = text_model(**enc)
            text_feat = out.hidden_states[-1][:, 0, :].squeeze(0)

        idx = random.randint(0, wear_tensor.shape[0] - 1)
        wear_row = wear_tensor[idx].to(device)

        fused_input = torch.cat((text_feat, wear_row)).unsqueeze(0)

        with torch.no_grad():
            logits = fused_model(fused_input)
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]

        label = "HIGH RISK" if probs[0] > probs[1] else "LOW RISK"

        md = (
            f"## Multimodal Depression Prediction\n\n"
            f"**Prediction:** `{label}`\n\n"
            f"- High Risk: `{probs[0]:.4f}`\n"
            f"- Low Risk: `{probs[1]:.4f}`\n\n"
            f"_Wearable sample index: {idx}_"
        )

        unscaled = try_get_unscaled_df(idx, uploaded_df)
        scaled = wearable_scaled_df(wear_row.cpu())

        return md, (unscaled if unscaled is not None else pd.DataFrame()), scaled

    except Exception:
        return "❌ Error\n```\n" + traceback.format_exc()[-800:] + "\n```", pd.DataFrame(), pd.DataFrame()

# ---------------------------
# GRADIO UI
# ---------------------------
iface = gr.Interface(
    fn=fused_predict_show_wearable,
    inputs=[
        gr.Textbox(lines=5, label="Social Media Text"),
        gr.File(label="Upload Wearable CSV (optional)")
    ],
    outputs=[
        gr.Markdown(label="Prediction"),
        gr.Dataframe(label="Unscaled Wearables"),
        gr.Dataframe(label="Scaled Wearables")
    ],
    title="DEPRESSION PREDICTION MODEL",
    description="Multimodal (Text + Wearables) — Colab"
)

iface.launch(share=True)


In [ ]:
import os
import torch
import re
import random
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ----------------------------
# Device
# ----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ----------------------------
# Text cleaning (UNCHANGED)
# ----------------------------
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ----------------------------
# Fused DNN (MUST MATCH TRAINING)
# ----------------------------
class FusedDNNClassifier(nn.Module):
    def __init__(self, input_dim):
        super(FusedDNNClassifier, self).__init__()
        self.layer_1 = nn.Linear(input_dim, 128)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)
        self.layer_2 = nn.Linear(128, 128)
        self.relu2 = nn.ReLU()
        self.layer_3 = nn.Linear(128, 2)

    def forward(self, x):
        x = self.layer_1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.layer_2(x)
        x = self.relu2(x)
        x = self.layer_3(x)
        return x

# ----------------------------
# Paths
# ----------------------------
SAVE_PATH = "/content/drive/MyDrive/depjct/depjct_outputs_2"
SAVE1_PATH = "/content/drive/MyDrive/depjct_outputs"
TOKENIZER_PATH = os.path.join(SAVE1_PATH, "twhin_bert_tokenizer")
TEXT_MODEL_PATH = os.path.join(SAVE1_PATH, "twhin_bert_sentiment_model")
WEARABLE_PATH = os.path.join(SAVE_PATH, "module2_wearable_features.pt")
FUSED_MODEL_PATH = os.path.join(SAVE_PATH, "fused_dnn_classifier_model.pt")

# ----------------------------
# Load TwHIN (FEATURE EXTRACTION ONLY)
# ----------------------------
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_PATH, fix_mistral_regex=True)
text_model = AutoModelForSequenceClassification.from_pretrained(
    TEXT_MODEL_PATH,
    output_hidden_states=True
).to(device)
text_model.eval()

print("✅ TwHIN-BERT loaded (feature extractor)")

# ----------------------------
# Load wearable features
# ----------------------------
wearable_features = torch.load(WEARABLE_PATH)
print(f"✅ Wearable features loaded: {wearable_features.shape}")

# ----------------------------
# Load fused model
# ----------------------------
INPUT_DIM = text_model.config.hidden_size + wearable_features.shape[1]

fused_model = FusedDNNClassifier(INPUT_DIM)
fused_model.load_state_dict(torch.load(FUSED_MODEL_PATH, map_location=device))
fused_model.to(device)
fused_model.eval()

print("✅ Fused multimodal model loaded")

# ======================================================
# 🔮 FUSED MULTIMODAL PREDICTION
# ======================================================
print("\n--- FUSED MODEL PREDICTION ---")

# User input
raw_text = input("Enter social media text: ")
text = clean_text(raw_text)

# ---- Text → Embedding (CLS token) ----
with torch.no_grad():
    enc = tokenizer(
        text,
        max_length=128,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    ).to(device)

    outputs = text_model(**enc)
    text_features = outputs.hidden_states[-1][:, 0, :]  # [CLS]

# ---- Simulated wearable input ----
idx = random.randint(0, wearable_features.shape[0] - 1)
wearable_sample = wearable_features[idx].unsqueeze(0).to(device)

# ---- Fuse ----
fused_input = torch.cat([text_features, wearable_sample], dim=1)

# ---- Predict ----
with torch.no_grad():
    logits = fused_model(fused_input)
    probs = F.softmax(logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

# ---- Output ----
print("\n--- RESULT ---")
print(f"Prediction : {'Higher Risk' if pred == 1 else 'Lower Risk'}")
print(f"Probabilities → Low Risk: {probs[0][0]:.4f}, High Risk: {probs[0][1]:.4f}")
